# Hill of Towie: Rekonstruktion des Wake-Steering-Zeitreihenplots (T03 → T07)

**Ziel:** Das obere Panel der Referenzabbildung *"Hill of Towie — T03 wake steering for T07 (T01 reference) — 2026-02-20 00:00–04:20 UTC"* aus der RES-Präsentation (WindEurope Tech 2026, Folie 13) mit den öffentlichen Daten nachbauen.

Die Referenzabbildung stammt aus dem Repo-Skript `scripts/wfc_analysis_2026/zxtm_wake_steer_animation.py`.

**Was der Plot zeigt:**
- Windrichtung (gepunktet) und Gondelposition (durchgezogen) von T01, T03 und T07, jeweils bezogen auf den geografischen Norden
- Die Phasen, in denen der Dynamic-Yaw-Controller Wake Steering an T03 aktiviert hat (grün) bzw. deaktiviert hat (rot)

**Datenquelle:** Hill of Towie wind farm open dataset v2 auf Zenodo (Record 20204946), Paket `turbine_fastlog.zip` (hochaufgelöste Turbinen- und Controller-Daten, Jan–Apr 2026).

Dieses Notebook ist so aufgebaut, dass es nach einem Kernel-Neustart von oben nach unten durchläuft und alle Variablen neu definiert.

## Schritt 0: Voraussetzungen

- Die Python-Umgebung wurde mit `uv sync` aus `pyproject.toml` / `uv.lock` aufgebaut, und dieses Notebook läuft mit dem Kernel dieser Umgebung. Dadurch ist das Repo-eigene Paket `hot_open` (Ordner `src/hot_open`) importierbar.
- **Beim allerersten Ausführen** lädt Schritt 3 das komplette Paket `turbine_fastlog.zip` von Zenodo herunter (über 10 GB) und entpackt es. Während des Entpackens wird ungefähr der doppelte Speicherplatz benötigt. Ein abgebrochener Download wird beim nächsten Ausführen fortgesetzt.
- Die Daten landen standardmäßig unter `~/temp/hill-of-towie-open-source-analysis/data` (festgelegt in `src/hot_open/settings.py`, änderbar über die Umgebungsvariable `HOT_OPEN_DATA_DIR`).
- Die Ladefunktion speichert jeden geladenen Tag zusätzlich als Parquet-Datei im Cache. Ab dem zweiten Durchlauf ist das Laden daher schnell.

Die Logging-Konfiguration sorgt dafür, dass die Meldungen der `hot_open`-Funktionen (Download-Fortschritt, gelesene Dateien, Cache-Treffer) im Notebook sichtbar sind.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from hot_open.fastlog_helpers import load_hot_fl_data
from hot_open.settings import get_filestore_dir

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## Schritt 1: Zeitraum festlegen

Wir wählen denselben Zeitraum wie die Referenzabbildung.

- Alle Zeitstempel im Datensatz sind **UTC**. Die Ladefunktionen verlangen deshalb Zeitstempel mit expliziter Zeitzone (`tz="UTC"`).
- Das Ende ist **exklusiv** (`end_dt_excl`): Daten ab 04:30 sind nicht mehr enthalten. Das ist dieselbe Logik wie `for (i = 0; i < n; i++)` in C++.

In [ ]:
start = pd.Timestamp("2026-02-20 00:00", tz="UTC")
end_excl = pd.Timestamp("2026-02-20 04:30", tz="UTC")

## Schritt 2: Die richtigen Signale auswählen

Das ist der entscheidende Schritt, und hier lag der Hauptfehler beim ersten Versuch.

### Problem: Jede Turbine hat ihren eigenen "Norden"

Die Siemens-Signale der Turbinen (z. B. `Wtc_TDI_YawPos_Value`, `Wtc_TDI_AcWindDr_Value`) liegen im **"turbine north"**-System: Jede Turbine misst ihre Gondelposition ab einem eigenen, bei der Inbetriebnahme gesetzten Nullpunkt, der vom geografischen Norden deutlich abweichen kann. Beim ersten Plot mit diesen Signalen lagen T01, T03 und T07 deshalb bei ca. 225°, 10° und 158°, obwohl alle drei denselben Wind sehen.

Die dazugehörigen Korrekturen ("northing corrections") sind im Repo in `scripts/wfc_analysis_2026/wind_up_config/northing_corrections/HOT_DY_Design_northing.yaml` hinterlegt. Details dazu im Anhang am Ende dieses Notebooks.

### Lösung: Die Tags des Dynamic-Yaw-Controllers verwenden

Laut Data Dictionary liegen alle Fastlog-Tags, deren Name auf `_true_degrees` endet, bereits im **True-North**-System. Diese Tags gibt es für alle 21 Turbinen (die Siemens-Tags nur für 9). Genau diese beiden verwendet auch das Referenzskript:

| Tag | Bedeutung |
|---|---|
| `computed_driver_pre_processed_yaw_direction_true_degrees` | Gemessene Gondelposition, mit der Nordkorrektur des Controllers. True North. |
| `computed_core_post_processed_consensus_wind_direction_true_degrees` | Windrichtungsschätzung des Controllers pro Turbine, gebildet aus den Daten mehrerer Turbinen ("consensus"). True North. |
| `computed_driver_post_processed_toggle_state` | Validierungs-Toggle des Dynamic-Yaw-Controllers: 0 = aus, 1 = an. Liefert die grün/roten Phasen. |
| `computed_core_post_processed_core_wake_steering_offset_degrees` | Kommandierter Wake-Steering-Offset, positiv = gegen den Uhrzeigersinn von oben gesehen. Wird für diesen Plot nicht gebraucht, aber später für PyWake. |

Die Tag-Namen legen wir als Konstanten ab, damit wir sie nicht mehrfach ausschreiben müssen (Tippfehler in so langen Strings fallen sonst erst beim Plotten auf).

In [ ]:
YAW = "computed_driver_pre_processed_yaw_direction_true_degrees"
WD = "computed_core_post_processed_consensus_wind_direction_true_degrees"
TOGGLE = "computed_driver_post_processed_toggle_state"
WS_OFFSET = "computed_core_post_processed_core_wake_steering_offset_degrees"

dy_tags = [YAW, WD, TOGGLE, WS_OFFSET]

# Turbinen aus der Referenzabbildung: T01 = Referenz, T03 = steuernde Turbine, T07 = profitierende Turbine
wtg_numbers = [1, 3, 7]
colors = {"T01": "tab:blue", "T03": "tab:orange", "T07": "tab:green"}

## Schritt 3: Fastlog-Daten laden

Wir verwenden die Funktion `load_hot_fl_data` aus `src/hot_open/fastlog_helpers.py`.

**Was die Funktion macht:**
- Beim ersten Aufruf lädt sie `turbine_fastlog.zip` von Zenodo herunter, entpackt es und löscht die ZIP-Datei danach.
- Sie lädt immer die Standard-Siemens-Tags (`SIEMENS_TAGS`) und zusätzlich alles, was über `extra_tags` übergeben wird.
- Die Rohdaten werden "on change" geloggt (nur wenn sich ein Wert ändert). Die Funktion bringt alles auf ein festes 1-Sekunden-Raster. Richtungen werden dabei mit **zirkulärer Mittelung** behandelt, denn der normale Mittelwert von 359° und 1° wäre 180° statt 0°.

**Stolperstein:** Der Parameter heißt zwar `data_dir`, wird intern aber als `filestore_dir` weitergereicht. Die Funktion sucht die Dateien unter `<filestore_dir>/FL/HOT/...`. Wir müssen also `get_filestore_dir()` übergeben, **nicht** `get_data_dir()`, sonst findet sie nach dem Download keine Dateien.

In [ ]:
fl = load_hot_fl_data(
    data_dir=get_filestore_dir(),  # Filestore-Ordner, nicht get_data_dir()!
    wtg_numbers=wtg_numbers,
    start_dt=start,
    end_dt_excl=end_excl,
    extra_tags=dy_tags,
)
fl.shape

### Aufbau des Ergebnisses

`fl` ist ein DataFrame mit:
- **Zeilen:** ein Zeitstempel pro Sekunde (UTC)
- **Spalten:** zwei Ebenen (`MultiIndex`), oben der Turbinenname, darunter der Tag. Eine einzelne Spalte spricht man als Tupel an, z. B. `fl[("T03", YAW)]`. Mit `fl["T03"]` bekommt man alle Tags einer Turbine als normales DataFrame.

Vergleichbar mit einer verschachtelten `std::map<std::string, std::map<std::string, Spalte>>` in C++.

Kurzer Plausibilitätscheck: Alle drei Turbinen sollten Windrichtungen und Gondelpositionen im Bereich von etwa 185–230° haben, und der Toggle-Zustand sollte nur die Werte 0 und 1 enthalten.

In [ ]:
for name in colors:
    print(name, "WD:", fl[(name, WD)].min().round(1), "bis", fl[(name, WD)].max().round(1),
          "| YAW:", fl[(name, YAW)].min().round(1), "bis", fl[(name, YAW)].max().round(1))

print("Toggle-Werte T03:", sorted(fl[("T03", TOGGLE)].dropna().unique()))

## Schritt 4: Den Plot erstellen

- **Linien:** Pro Turbine eine gepunktete Linie für die Windrichtung und eine durchgezogene für die Gondelposition, jeweils in derselben Farbe.
- **Hintergrundflächen:** `fill_between` mit `where=(toggle == 1)` füllt nur dort, wo der Toggle an ist. `toggle == 1` erzeugt ein Array aus `True`/`False`, eines pro Sekunde. Zeitschritte ohne Toggle-Wert (NaN) ergeben `False` und bleiben ungefüllt.
- **`transform=ax.get_xaxis_transform()`:** Die x-Werte bleiben Zeitstempel, die y-Werte `0` und `1` werden aber als Anteil der Achsenhöhe interpretiert (0 = unterer Rand, 1 = oberer Rand). So füllen die Flächen immer die ganze Höhe, unabhängig vom Wertebereich der Richtungen.
- **`ax.margins(x=0)`:** Entfernt den automatischen weißen Rand links und rechts, sodass die Achse genau beim ersten und letzten Datenpunkt beginnt und endet.

Die Steering-Phasen beziehen sich auf T03, weil T03 die steuernde Turbine ist.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for name, c in colors.items():
    ax.plot(fl.index, fl[(name, WD)], color=c, ls=":", alpha=0.6, label=f"{name} wind dir")
    ax.plot(fl.index, fl[(name, YAW)], color=c, label=f"{name} yaw pos")

toggle = fl[("T03", TOGGLE)]
ax.fill_between(fl.index, 0, 1, where=(toggle == 1), transform=ax.get_xaxis_transform(),
                color="tab:green", alpha=0.15, label="steering on")
ax.fill_between(fl.index, 0, 1, where=(toggle == 0), transform=ax.get_xaxis_transform(),
                color="tab:red", alpha=0.15, label="steering off")

ax.margins(x=0)
ax.set_ylabel("Richtung [°N]")
ax.set_title("Hill of Towie – T03 wake steering for T07: Fastlog-Daten (True North)")
ax.grid(True, alpha=0.3)
ax.legend(ncol=4, fontsize=8)
plt.show()

### Was man im Plot sieht

- In den **roten Phasen** folgen alle drei Turbinen dem Wind: Gondelposition und Windrichtung liegen übereinander.
- In den **grünen Phasen** weicht die Gondelposition von **T03** (orange, durchgezogen) gezielt von der Windrichtung ab, teilweise um mehr als 10°. Das ist das Wake Steering: T03 dreht sich bewusst schräg, damit ihr Nachlauf an T07 vorbeigelenkt wird.
- T01 (Referenz) und T07 folgen auch in den grünen Phasen normal dem Wind.

## Anhang: Warum die True-North-Tags? Kontrolle mit den Nordkorrekturen

Die Standard-Siemens-Tags (`YawPos_Value`, `AcWindDr_Value`) sind bereits in `fl` enthalten, weil `load_hot_fl_data` sie immer mitlädt. Damit lässt sich prüfen, dass die True-North-Tags tatsächlich den rohen Werten plus Nordkorrektur entsprechen.

Die Korrekturen stehen in `scripts/wfc_analysis_2026/wind_up_config/northing_corrections/HOT_DY_Design_northing.yaml` als `[Turbine, gültig ab, Korrektur in Grad]`. Sie ändern sich über die Zeit (z. B. nach einer Neukalibrierung). Für den 20.02.2026 gilt jeweils der letzte Eintrag vor diesem Datum:

| Turbine | Gültig ab | Korrektur |
|---|---|---|
| T01 | 2019-10-14 23:50 | −23.57° |
| T03 | 2025-08-07 13:10 | −177.46° |
| T07 | 2025-08-29 07:20 | +39.44° |

Umrechnung: **wahre Richtung = (Turbinenrichtung + Korrektur) mod 360**

Beim Vergleich zweier Winkel muss man ebenfalls zirkulär rechnen: Die Differenz zwischen 359° und 1° ist 2°, nicht 358°. Der Ausdruck `(a - b + 180) % 360 - 180` bringt jede Differenz in den Bereich −180° bis +180°.

In [ ]:
# Aus HOT_DY_Design_northing.yaml, gültig am 2026-02-20
northing_correction = {"T01": -23.56751060485834, "T03": -177.46022644042966, "T07": 39.44209289550781}

def circ_diff(a, b):
    # Zirkuläre Differenz a - b im Bereich [-180, 180)
    return (a - b + 180) % 360 - 180

fig, ax = plt.subplots(figsize=(14, 3))
for name, c in colors.items():
    yaw_corrected = (fl[(name, "YawPos_Value")] + northing_correction[name]) % 360
    diff = circ_diff(yaw_corrected, fl[(name, YAW)])
    ax.plot(fl.index, diff, color=c, label=name)
    print(f"{name}: mittlere Abweichung {diff.mean():.2f}°, maximale Abweichung {diff.abs().max():.2f}°")

ax.margins(x=0)
ax.set_ylabel("Abweichung [°]")
ax.set_title("Rohe Gondelposition + YAML-Korrektur minus True-North-Tag")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

Liegen die Abweichungen nahe 0°, bestätigt das: Die True-North-Tags sind die rohen Turbinensignale mit genau diesen Nordkorrekturen. Kleine Ausreißer können durch das unterschiedliche Logging der Signale entstehen (beide Signale werden unabhängig voneinander "on change" aufgezeichnet).

**Wichtig für spätere Analysen:** Auch die 10-Minuten-SCADA-Felder (z. B. `wtc_NacelPos_mean`) liegen im Turbine-North-System. Wer sie für Richtungsanalysen oder als Eingabe für PyWake verwendet, muss die Korrekturen aus der YAML-Datei selbst anwenden.

## Nächste Schritte

1. Die Fastlog-Daten auf etwa 1 Minute mitteln (zirkulär für Richtungen, z. B. mit `circ_mean_resample_degrees` aus `hot_open.circular_math`). Für PyWake ist 1 Hz physikalisch nicht sinnvoll, weil jeder Zeitschritt als stationärer Zustand behandelt wird, die Luft von T03 bis T07 aber rund eine Minute unterwegs ist.
2. Eine PyWake-Zeitreihensimulation mit der Consensus-Windrichtung als Anströmung aufsetzen.
3. Den Gierwinkel von T03 an PyWake übergeben (Argument `yaw`, positiv = gegen den Uhrzeigersinn, gleiche Konvention wie der Wake-Steering-Offset des Controllers) und mit Deflection-Modell rechnen.